# Lecture 08: Darboux-Moser-Weinstein Theory

**Source span.** Printed pages 44-48; physical PDF pages 58-62 in `Lectures on Symplectic Geometry.pdf`. I inspected the local PDF text for this span before revising the notebook.

**Lecture goal.** Build the local-normal-form chain that begins with the Classical Darboux theorem, passes through Lagrangian subspaces, and ends at the Weinstein Lagrangian neighborhood theorem and its coisotropic extension.

The lecture has one recurring proof move: reduce a local symplectic claim to a linear algebra anchor, then use the Moser local theorem to move from the anchored model to the actual form. Darboux uses the anchor at a point. Weinstein uses the anchor along a Lagrangian submanifold. The coisotropic statement keeps the same philosophy but changes the condition on the restricted form and characteristic directions.

The notebook treats that move as something inspectable: matrices normalize the form, graph complements repair a non-Lagrangian complement, cotangent graphs test `d alpha = 0`, proof diagrams show where Whitney extension and Moser correction enter, and small symbolic checks cover the oriented-surface homework themes without copying exercise text.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, read_json, save_json, save_matplotlib
from utils.symplectic import lagrangian_residual, standard_omega

ARTIFACT_TOPIC = "lecture-08"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact topic: {ARTIFACT_TOPIC}")

## Translation Guide And Library Routing

| Source idea | Computational object | Check |
| --- | --- | --- |
| Classical Darboux theorem | a nonstandard skew matrix `Omega_local` and a chart matrix `B` | `B.T @ Omega_local @ B` equals the standard matrix |
| Darboux chart | a coordinate change centered at the chosen point | the pulled-back two-form has the standard block form |
| Lagrangian subspaces | half-dimensional bases with zero pulled-back two-form | `basis.T @ Omega @ basis` vanishes |
| arbitrary complement to a Lagrangian | graph `q = C p` over the complementary variables | the skew part of `C` measures failure |
| cotangent model | graph of a one-form over base coordinates | the graph is Lagrangian exactly when `d alpha = 0` |
| local normal form | point or submanifold model plus Moser correction | linear residual and proof-dependency ledger both close |
| Weinstein Lagrangian neighborhood theorem | tangent-level maps `L_p`, Whitney extension, then Moser local theorem | the map fixes `X` and pulls one form to the other |
| coisotropic extension | subspace with its symplectic orthogonal contained in it | `U^omega subset U` and restricted forms match |

**Library routing.** `numpy` is the right level for the linear symplectic matrix checks and graph complements. `matplotlib` gives durable diagrams for the Darboux chart, Lagrangian repair, and cotangent graph tests. `networkx` is used only for proof dependency structure. `sympy` is reserved for exact checks where the lecture homework mentions surface area forms and four-dimensional convex-combination behavior.

## Visualization Storyboard

1. **Darboux linear chart normalization.** Start with a nonstandard local two-form built from a coordinate matrix. The visual shows distorted symplectic coordinate circles and the residual verifies that the Darboux chart returns the standard form.
2. **Lagrangian complement and cotangent graph test.** A graph complement is repaired by symmetrizing the matrix. A cotangent graph is Lagrangian when the one-form is closed; the comparison graph makes the curl obstruction visible.
3. **Weinstein proof and coisotropic extension.** A proof graph separates the pointwise linear algebra, Whitney extension, and Moser correction. A small subspace calculation records the coisotropic condition used by the later generalization.
4. **Homework-facing checks.** The sphere area-form identity in cylindrical coordinates and the four-dimensional convex-combination failure are checked symbolically/numerically as compact code snippets.

In [ ]:
# Darboux chart as a linear normalization problem.
J = standard_omega(2)
A = np.array([
    [1.2, 0.25, 0.15, 0.0],
    [0.05, 0.9, 0.0, -0.2],
    [0.1, 0.0, 1.1, 0.35],
    [0.0, -0.15, 0.2, 0.85],
], dtype=float)
Omega_local = A.T @ J @ A
B = np.linalg.inv(A)
darboux_residual = float(np.linalg.norm(B.T @ Omega_local @ B - J))
nondegenerate_margin = float(abs(np.linalg.det(Omega_local)))
assert darboux_residual < 1e-12
assert nondegenerate_margin > 1e-6

theta = np.linspace(0, 2 * np.pi, 240)
plane_1 = np.vstack([np.cos(theta), np.zeros_like(theta), np.sin(theta), np.zeros_like(theta)])
plane_2 = np.vstack([np.zeros_like(theta), np.cos(theta), np.zeros_like(theta), np.sin(theta)])
distorted_1 = A @ plane_1
distorted_2 = A @ plane_2

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
for ax, standard, distorted, title, coords in [
    (axes[0], plane_1, distorted_1, "first symplectic plane", (0, 2)),
    (axes[1], plane_2, distorted_2, "second symplectic plane", (1, 3)),
]:
    ax.plot(standard[coords[0]], standard[coords[1]], color="#2c7fb8", lw=2, label="standard Darboux circle")
    ax.plot(distorted[coords[0]], distorted[coords[1]], color="#d95f02", lw=2, label="same circle before chart")
    ax.axhline(0, color="0.8", lw=0.8)
    ax.axvline(0, color="0.8", lw=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("x-coordinate")
    ax.set_ylabel("y-coordinate")
    ax.legend(loc="upper right", fontsize=8)
fig.suptitle("Darboux theorem: a local chart normalizes the two-form at the anchor point")
darboux_chart_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "darboux-linear-chart-normalization.png")
plt.close(fig)
display_artifact(darboux_chart_path, width=760)
print({"darboux_residual": darboux_residual, "det_Omega_local": nondegenerate_margin})

In [ ]:
# Lagrangian complement repair and cotangent graph obstruction.
def graph_basis(C):
    return np.vstack([C, np.eye(C.shape[0])])

C = np.array([[0.20, 1.00], [-0.40, 0.30]], dtype=float)
C_repaired = 0.5 * (C + C.T)
residual_complement_before = lagrangian_residual(graph_basis(C), J)
residual_complement_after = lagrangian_residual(graph_basis(C_repaired), J)
assert residual_complement_before > 1e-2
assert residual_complement_after < 1e-12

q1 = np.linspace(-1.2, 1.2, 17)
q2 = np.linspace(-1.2, 1.2, 17)
Q1, Q2 = np.meshgrid(q1, q2)
alpha_exact_1 = 0.30 * Q1 + 0.10 * Q2
alpha_exact_2 = 0.10 * Q1 - 0.20 * Q2
alpha_bad_1 = -Q2
alpha_bad_2 = Q1
curl_exact = 0.10 - 0.10
curl_bad = 1.0 - (-1.0)
assert abs(curl_exact) < 1e-12
assert abs(curl_bad - 2.0) < 1e-12

fig, axes = plt.subplots(2, 2, figsize=(11, 8.2))
axes[0, 0].bar(["arbitrary W", "repaired W'"], [residual_complement_before, residual_complement_after], color=["#d95f02", "#1b9e77"])
axes[0, 0].set_ylabel("||B.T Omega B||")
axes[0, 0].set_title("Proposition 8.2: repair a complement to a Lagrangian one")
axes[0, 0].set_yscale("symlog", linthresh=1e-12)
axes[0, 0].annotate("skew part removed", xy=(1, max(residual_complement_after, 1e-12)), xytext=(0.55, residual_complement_before / 2), arrowprops={"arrowstyle": "->"})

im = axes[0, 1].imshow(C - C.T, cmap="coolwarm", vmin=-1.6, vmax=1.6)
axes[0, 1].set_title("obstruction matrix C - C.T")
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_yticks([0, 1])
fig.colorbar(im, ax=axes[0, 1], fraction=0.046)

for ax, A1, A2, title, color in [
    (axes[1, 0], alpha_exact_1, alpha_exact_2, "cotangent graph of an exact one-form: d alpha = 0", "#1b9e77"),
    (axes[1, 1], alpha_bad_1, alpha_bad_2, "rotating one-form: d alpha is nonzero", "#d95f02"),
]:
    ax.quiver(Q1, Q2, A1, A2, color=color, angles="xy")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_xlabel("base q1")
    ax.set_ylabel("base q2")
    ax.set_title(title)
fig.suptitle("Weinstein neighborhood model: Lagrangian means the pulled-back two-form vanishes")
lagrangian_cotangent_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "lagrangian-complement-and-cotangent-graphs.png")
plt.close(fig)
display_artifact(lagrangian_cotangent_path, width=760)
print({"before": residual_complement_before, "after": residual_complement_after, "bad_curl": curl_bad})

In [ ]:
# Proof dependency graph and a finite coisotropic check.
G = nx.DiGraph()
edges = [
    ("symplectic basis at p", "coordinates with omega_p standard"),
    ("coordinates with omega_p standard", "compare omega and standard form"),
    ("compare omega and standard form", "Moser local theorem on X={p}"),
    ("Moser local theorem on X={p}", "Darboux chart"),
    ("U = T_p X Lagrangian", "choose metric complement U_perp"),
    ("choose metric complement U_perp", "canonical Lagrangian complements"),
    ("canonical Lagrangian complements", "linear map L_p fixes T_p X"),
    ("linear map L_p fixes T_p X", "Whitney extension h"),
    ("Whitney extension h", "forms agree along X"),
    ("forms agree along X", "Moser local theorem along X"),
    ("Moser local theorem along X", "Weinstein Lagrangian neighborhood theorem"),
    ("equal restricted forms", "coisotropic embedding theorem"),
    ("U^omega subset U", "coisotropic embedding theorem"),
]
G.add_edges_from(edges)
pos = {
    "symplectic basis at p": (0, 3),
    "coordinates with omega_p standard": (1.6, 3),
    "compare omega and standard form": (3.2, 3),
    "Moser local theorem on X={p}": (4.8, 3),
    "Darboux chart": (6.4, 3),
    "U = T_p X Lagrangian": (0, 1.5),
    "choose metric complement U_perp": (1.4, 1.5),
    "canonical Lagrangian complements": (2.8, 1.5),
    "linear map L_p fixes T_p X": (4.2, 1.5),
    "Whitney extension h": (5.6, 1.5),
    "forms agree along X": (4.2, 0.3),
    "Moser local theorem along X": (5.6, 0.3),
    "Weinstein Lagrangian neighborhood theorem": (7.0, 0.9),
    "equal restricted forms": (1.0, -0.9),
    "U^omega subset U": (3.2, -0.9),
    "coisotropic embedding theorem": (5.4, -0.9),
}

U = np.eye(4)[:, [0, 1, 2]]  # hyperplane p2=0 in coordinates x1,x2,y1,y2
M = sp.Matrix(U.T @ J)
null_basis = M.nullspace()
coisotropic_dim = len(null_basis)
coisotropic_vectors = [np.array(vec, dtype=float).reshape(4) for vec in null_basis]
coisotropic_inside = all(abs(v[3]) < 1e-12 for v in coisotropic_vectors)
assert coisotropic_dim == 1
assert coisotropic_inside

fig, ax = plt.subplots(figsize=(12, 6.2))
colors = []
for node in G.nodes:
    if "Darboux" in node or node in {"symplectic basis at p", "coordinates with omega_p standard", "compare omega and standard form", "Moser local theorem on X={p}"}:
        colors.append("#8dd3c7")
    elif "coisotropic" in node or node in {"equal restricted forms", "U^omega subset U"}:
        colors.append("#bebada")
    else:
        colors.append("#ffffb3")
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=16, width=1.4, edge_color="0.35")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=2400, linewidths=1.0, edgecolors="0.25")
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.text(0.05, -1.65, "Finite coisotropic check: U={p2=0}; dim(U^omega)=1 and U^omega is spanned inside U.", fontsize=10, bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "0.6"})
ax.set_axis_off()
ax.set_title("Darboux and Weinstein proof ledger: linear anchor, extension, Moser correction")
proof_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "weinstein-proof-and-coisotropic-extension.png")
plt.close(fig)
display_artifact(proof_path, width=760)
print({"coisotropic_dim": coisotropic_dim, "inside_U": coisotropic_inside})

In [ ]:
# Compact code snippets for the source span's homework-facing themes.
theta_sym, z_sym = sp.symbols("theta z", real=True)
r = sp.sqrt(1 - z_sym**2)
u = sp.Matrix([r * sp.cos(theta_sym), r * sp.sin(theta_sym), z_sym])
sphere_area_density = sp.simplify(u.dot(u.diff(theta_sym).cross(u.diff(z_sym))))
assert sp.simplify(sphere_area_density - 1) == 0

# A two-form matrix in R4: omega0 and -omega0 induce the same orientation, but their midpoint is degenerate.
omega0 = J
omega1 = -J
convex_midpoint_det = float(np.linalg.det(0.5 * (omega0 + omega1)))
assert abs(convex_midpoint_det) < 1e-12

K = np.array([
    [0, 1, 0, 0],
    [-1, 0, 0, 0],
    [0, 0, 0, -1],
    [0, 0, 1, 0],
], dtype=float)
ts = np.linspace(0, 1, 101)
path_dets = [float(np.linalg.det(np.cos(np.pi * t) * J + np.sin(np.pi * t) * K)) for t in ts]
path_min_det = min(abs(value) for value in path_dets)
assert path_min_det > 0.99

homework_checks = {
    "sphere_area_density_in_theta_z": str(sphere_area_density),
    "convex_midpoint_det_R4": convex_midpoint_det,
    "path_min_abs_det_from_omega_to_minus_omega": path_min_det,
    "interpretation": "In dimension two, positive area densities stay nonzero under convex interpolation; in dimension four, same orientation does not protect the convex midpoint.",
}
print(homework_checks)

## Reading The Visuals

The Darboux figure is the point case. The orange curves are what standard coordinate circles look like before the chart has normalized the form; the residual says the chart matrix really pulls the local form back to the standard Darboux matrix.

The Lagrangian-complement figure is the algebra inside Propositions 8.2 and 8.3. A complement to `U` can fail to be Lagrangian by a skew obstruction. Removing that obstruction builds the Lagrangian complement. The cotangent panels translate the Weinstein neighborhood theorem into the model `T^*X`: exact or closed one-form graphs have zero pulled-back symplectic form, while a rotating one-form exposes nonzero `d alpha`.

The proof graph keeps the three theorems separate. Darboux anchors at one point; Weinstein anchors along a Lagrangian submanifold and uses Whitney extension before Moser; the coisotropic embedding theorem asks for equal restricted forms and the containment `U^omega subset U`. These are the conditions a reader should validate before applying a local normal form.

In [ ]:
residuals = {
    "darboux_chart_residual": darboux_residual,
    "local_form_det_abs": nondegenerate_margin,
    "lagrangian_complement_residual_before": residual_complement_before,
    "lagrangian_complement_residual_after": residual_complement_after,
    "exact_cotangent_graph_curl": curl_exact,
    "bad_cotangent_graph_curl": curl_bad,
    "coisotropic_orthogonal_dimension": coisotropic_dim,
    "coisotropic_orthogonal_inside_U": coisotropic_inside,
    "sphere_area_density_in_theta_z": str(sphere_area_density),
    "convex_midpoint_det_R4": convex_midpoint_det,
    "path_min_abs_det_from_omega_to_minus_omega": path_min_det,
}
residual_path = save_json(residuals, ARTIFACT_TOPIC, "checks", "darboux-weinstein-residuals.json")

source_span = {
    "title": "Darboux-Moser-Weinstein Theory",
    "printed_pages": "44-48",
    "physical_pdf_pages": "58-62",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "sections": [
        "Classical Darboux theorem",
        "Lagrangian subspaces",
        "Weinstein Lagrangian neighborhood theorem",
        "Whitney extension theorem sketch",
        "Coisotropic Embedding Theorem",
        "oriented-surface homework themes",
    ],
    "concepts": [
        "Darboux chart",
        "Lagrangian subspace",
        "cotangent model",
        "local normal form",
        "coisotropic extension",
    ],
    "copyright_note": "Source pages were used for terminology and coverage; notebook prose, visuals, and code are original and no page crops or long exercise text are copied.",
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Make local symplectic normal forms inspectable through matrix normalization, Lagrangian graph tests, and proof-dependency diagrams.",
    "source_span_read": source_span,
    "library_routing": [
        {"concept": "Darboux chart", "representation": "matrix normalization and distorted coordinate circles", "library": "numpy + matplotlib", "why": "the theorem is local and the residual is a direct matrix identity"},
        {"concept": "Lagrangian subspace", "representation": "graph complement residual and repair", "library": "numpy + matplotlib", "why": "the source proof is finite-dimensional linear algebra"},
        {"concept": "cotangent model", "representation": "one-form graph and curl obstruction", "library": "numpy + matplotlib", "why": "closedness of the one-form is visible as a Lagrangian pullback test"},
        {"concept": "Weinstein proof", "representation": "dependency graph", "library": "networkx", "why": "the proof uses several theorem moves that should not be collapsed"},
        {"concept": "homework checks", "representation": "exact area-form and R4 determinant checks", "library": "sympy + numpy", "why": "symbolic and numeric snippets catch the stated invariants compactly"},
    ],
    "visual_sequence": [
        {"concept": "Darboux chart", "artifact": "artifacts/lecture-08/figures/darboux-linear-chart-normalization.png", "inspection_target": "the chart residual returns the local two-form to the standard block matrix"},
        {"concept": "Lagrangian complement and cotangent model", "artifact": "artifacts/lecture-08/figures/lagrangian-complement-and-cotangent-graphs.png", "inspection_target": "the repaired complement and exact graph have zero Lagrangian residual"},
        {"concept": "Weinstein and coisotropic proof ledger", "artifact": "artifacts/lecture-08/figures/weinstein-proof-and-coisotropic-extension.png", "inspection_target": "Whitney extension and Moser correction appear as distinct proof moves"},
    ],
    "computational_checks": residuals,
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")

final_sanity = {
    "artifacts": [
        str(darboux_chart_path.relative_to(BOOK_ROOT)),
        str(lagrangian_cotangent_path.relative_to(BOOK_ROOT)),
        str(proof_path.relative_to(BOOK_ROOT)),
        str(residual_path.relative_to(BOOK_ROOT)),
        str(source_path.relative_to(BOOK_ROOT)),
        str(storyboard_path.relative_to(BOOK_ROOT)),
    ],
    "assertions": {
        "darboux_chart_residual_lt_1e-12": darboux_residual < 1e-12,
        "lagrangian_repair_residual_lt_1e-12": residual_complement_after < 1e-12,
        "bad_cotangent_graph_has_nonzero_curl": abs(curl_bad) > 1.0,
        "coisotropic_orthogonal_inside_U": coisotropic_inside,
        "sphere_area_form_is_dtheta_wedge_dz": sphere_area_density == 1,
        "R4_convex_midpoint_degenerate": abs(convex_midpoint_det) < 1e-12,
        "R4_nonconvex_path_stays_symplectic": path_min_det > 0.99,
    },
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print(json.dumps(final_sanity, indent=2))

In [ ]:
final_sanity = read_json(ARTIFACT_ROOT / "checks" / "final-sanity.json")
assert all(final_sanity["assertions"].values())
for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    assert path.exists(), relative
    assert path.stat().st_size > 100, relative

for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    if path.suffix.lower() in {".png", ".html", ".svg"}:
        display_artifact(path, width=760, height=430 if path.suffix == ".html" else None)

print("Lecture 08 checks passed; artifacts are present and nonempty.")

## Takeaways

- Darboux's theorem is a local coordinate-normalization theorem: after anchoring the form at a point, Moser's local theorem upgrades the linear standard form to a neighborhood chart.
- A Lagrangian subspace is not just isotropic; it is half-dimensional, and the zero pullback of the two-form is the check that makes the definition computational.
- Weinstein's Lagrangian neighborhood theorem says that a neighborhood of a Lagrangian is modeled by its cotangent bundle; closed one-form graphs are the local test objects.
- The coisotropic embedding theorem keeps the local-normal-form strategy but replaces the Lagrangian condition with equal restricted forms and characteristic-direction containment.
- The homework themes reinforce the boundary of intuition: surface area forms are locally Darboux, but in four dimensions a convex combination of same-orientation symplectic forms can still become degenerate.